# 🎙️ SignalPath — Real-Time Caption Trust Scoring
### Accessibility Gap Detection for Live Events

**What this notebook does:**
1. Downloads & prepares datasets (LibriSpeech + TED-LIUM) for training
2. Builds a synthetic error injection pipeline to generate labeled training data
3. Engineers features: word confidence, perplexity, WER per segment
4. Trains a LightGBM trust score classifier
5. Captions any YouTube URL or uploaded video file
6. Scores every caption segment — exact word-level error detection
7. Launches a Gradio dashboard with full output

**Runtime:** GPU (T4 recommended) — Runtime > Change runtime type > T4 GPU

---
## Section 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q openai-whisper
!pip install -q yt-dlp
!pip install -q jiwer
!pip install -q lightgbm
!pip install -q transformers accelerate
!pip install -q datasets
!pip install -q gradio
!pip install -q librosa soundfile
!pip install -q torch torchaudio
!apt-get install -q ffmpeg

print("✅ All packages installed.")

In [ ]:
import os
import re
import json
import math
import random
import subprocess
import warnings
import numpy as np
import pandas as pd
import torch
import whisper
import librosa
import soundfile as sf
import lightgbm as lgb
import gradio as gr

from pathlib import Path
from jiwer import wer, process_words
from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

warnings.filterwarnings('ignore')

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {DEVICE}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'None'}")

---
## Section 2 — Load ASR Model (Whisper)

In [ ]:
# Load Whisper — 'base' is fast and good enough for trust scoring
# Options: tiny | base | small | medium | large-v2
# On free Colab T4: base or small recommended

WHISPER_MODEL_SIZE = "base"  # Change to 'small' for better accuracy

print(f"Loading Whisper {WHISPER_MODEL_SIZE}...")
asr_model = whisper.load_model(WHISPER_MODEL_SIZE, device=DEVICE)
print(f"✅ Whisper {WHISPER_MODEL_SIZE} loaded on {DEVICE}")

---
## Section 3 — Load Perplexity Model (DistilGPT2)

Used to score how linguistically plausible each caption segment is.
Low plausibility = likely ASR hallucination or garbled output.

In [ ]:
print("Loading DistilGPT2 for perplexity scoring...")
ppl_tokenizer = GPT2TokenizerFast.from_pretrained("distilgpt2")
ppl_model = GPT2LMHeadModel.from_pretrained("distilgpt2").to(DEVICE)
ppl_model.eval()
print("✅ DistilGPT2 loaded")

def compute_perplexity(text: str) -> float:
    """Compute perplexity of a text segment using DistilGPT2.
    Higher perplexity = more unusual / likely wrong caption.
    Returns a normalized score capped at 500.
    """
    if not text or len(text.strip()) < 3:
        return 500.0
    try:
        encodings = ppl_tokenizer(text, return_tensors="pt").to(DEVICE)
        max_length = ppl_model.config.n_positions
        input_ids = encodings.input_ids[:, :max_length]
        with torch.no_grad():
            outputs = ppl_model(input_ids, labels=input_ids)
            loss = outputs.loss
        ppl = torch.exp(loss).item()
        return min(ppl, 500.0)
    except Exception:
        return 500.0

# Quick sanity check
normal = compute_perplexity("The president gave a speech about climate change today.")
garbled = compute_perplexity("Xqr flump wizzle bork the snarp klanged.")
print(f"Normal sentence perplexity:  {normal:.1f}")
print(f"Garbled sentence perplexity: {garbled:.1f}")
print("✅ Higher = worse (as expected)" if garbled > normal else "⚠ Check perplexity model")

---
## Section 4 — Dataset Loading

We use **LibriSpeech** (clean audiobook speech) and **TED-LIUM** (lecture-style speech).
Both are freely available on HuggingFace — no login required.
We stream them to avoid downloading gigabytes.

In [ ]:
# How many samples to pull for training data generation
# 50 samples per dataset is fast for demo; increase to 200+ for better classifier
N_LIBRISPEECH = 50
N_TEDLIUM = 30

print("Loading LibriSpeech (streaming)...")
librispeech_ds = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="validation",
    streaming=True,
    trust_remote_code=True
)

print("Loading TED-LIUM Release 3 (streaming)...")
tedlium_ds = load_dataset(
    "LIUM/tedlium",
    "release3",
    split="validation",
    streaming=True,
    trust_remote_code=True
)

print("✅ Both datasets loaded in streaming mode")

---
## Section 5 — Synthetic Error Injection + Training Data Pipeline

**The problem:** No public dataset of labeled live-caption errors exists.

**Our solution:**
1. Take clean audio + reference transcript from LibriSpeech / TED-LIUM
2. Run Whisper on the audio → get ASR output
3. Compare ASR output to reference → compute WER per segment → that's our error label
4. Extract features from the ASR output
5. Train LightGBM: features → trust score

In [ ]:
# ── Feature extraction from a Whisper segment ──────────────────────────────

JARGON_WORDS = set([
    # Medical
    "echocardiogram", "myocardial", "infarction", "angioplasty", "tachycardia",
    "arrhythmia", "oncology", "chemotherapy", "radiotherapy", "laparoscopic",
    # Legal
    "plaintiff", "defendant", "injunction", "subpoena", "jurisprudence",
    "habeas", "corpus", "affidavit", "deposition", "indictment",
    # Technical
    "kubernetes", "containerization", "microservices", "asynchronous",
    "cryptographic", "blockchain", "tokenization", "latency", "throughput",
    # Finance
    "quantitative", "derivatives", "arbitrage", "collateralization",
    "securitization", "amortization", "liquidity", "volatility"
])

def extract_features(text: str, word_confidences: list, audio_array: np.ndarray, sr: int) -> dict:
    """
    Extract features from a transcribed segment for trust scoring.
    
    Features:
      Acoustic: mean/min/std of word-level confidence, signal-to-noise ratio
      Linguistic: perplexity, avg word length, unique word ratio
      Domain: jargon word count, jargon ratio
      Structural: segment length, words per second, repetition ratio
    """
    words = text.strip().split()
    n_words = len(words) if words else 1

    # Acoustic confidence features
    confs = word_confidences if word_confidences else [0.5]
    conf_mean = float(np.mean(confs))
    conf_min  = float(np.min(confs))
    conf_std  = float(np.std(confs)) if len(confs) > 1 else 0.0
    low_conf_ratio = sum(1 for c in confs if c < 0.5) / len(confs)

    # Signal-to-noise ratio from audio
    if audio_array is not None and len(audio_array) > 0:
        signal_power = np.mean(audio_array ** 2)
        noise_floor  = np.percentile(np.abs(audio_array), 10) ** 2 + 1e-10
        snr_db = 10 * np.log10(signal_power / noise_floor + 1e-10)
        snr_db = float(np.clip(snr_db, -10, 60))
    else:
        snr_db = 20.0

    # Perplexity
    perplexity = compute_perplexity(text)
    log_perplexity = math.log(perplexity + 1)

    # Linguistic features
    avg_word_len  = np.mean([len(w) for w in words]) if words else 0
    unique_ratio  = len(set(words)) / n_words

    # Repetition: count repeated adjacent words
    repetitions = sum(1 for i in range(1, len(words)) if words[i].lower() == words[i-1].lower())
    repetition_ratio = repetitions / n_words

    # Domain jargon
    words_lower = [w.lower().strip('.,!?;:') for w in words]
    jargon_count = sum(1 for w in words_lower if w in JARGON_WORDS)
    jargon_ratio  = jargon_count / n_words

    # Structural
    segment_length = len(text)

    return {
        "conf_mean":        conf_mean,
        "conf_min":         conf_min,
        "conf_std":         conf_std,
        "low_conf_ratio":   low_conf_ratio,
        "snr_db":           snr_db,
        "log_perplexity":   log_perplexity,
        "avg_word_len":     float(avg_word_len),
        "unique_ratio":     float(unique_ratio),
        "repetition_ratio": float(repetition_ratio),
        "jargon_count":     float(jargon_count),
        "jargon_ratio":     float(jargon_ratio),
        "segment_length":   float(segment_length),
        "n_words":          float(n_words),
    }

print("✅ Feature extractor defined")

In [ ]:
# ── Transcribe one audio sample and return per-segment features + labels ───

def process_sample(audio_array: np.ndarray, sr: int, reference_text: str):
    """
    Given raw audio + reference transcript:
    1. Run Whisper with word timestamps
    2. For each segment, compute features + WER vs reference
    3. Label: is_error = 1 if segment WER > 0.2, else 0
    Returns list of (features_dict, label, segment_wer, segment_text)
    """
    # Resample to 16kHz for Whisper
    if sr != 16000:
        audio_array = librosa.resample(audio_array.astype(np.float32), orig_sr=sr, target_sr=16000)
        sr = 16000

    # Normalize
    audio_array = audio_array.astype(np.float32)
    if np.max(np.abs(audio_array)) > 0:
        audio_array /= np.max(np.abs(audio_array))

    # Transcribe with word-level timestamps and logprob (confidence proxy)
    try:
        result = asr_model.transcribe(
            audio_array,
            word_timestamps=True,
            verbose=False,
            temperature=0.0
        )
    except Exception as e:
        print(f"  Whisper error: {e}")
        return []

    records = []
    ref_clean = reference_text.lower().strip()

    for seg in result.get("segments", []):
        seg_text = seg.get("text", "").strip()
        if not seg_text or len(seg_text.split()) < 2:
            continue

        # Word-level confidence from Whisper logprobs
        words_data = seg.get("words", [])
        word_confs = []
        for w in words_data:
            lp = w.get("probability", None)
            if lp is None:
                lp = math.exp(w.get("logprob", -1.0))
            word_confs.append(float(np.clip(lp, 0.0, 1.0)))

        # Segment audio slice for SNR
        start_s = seg.get("start", 0.0)
        end_s   = seg.get("end", len(audio_array) / sr)
        start_i = int(start_s * sr)
        end_i   = int(end_s   * sr)
        seg_audio = audio_array[start_i:end_i] if end_i > start_i else audio_array

        # Features
        feats = extract_features(seg_text, word_confs, seg_audio, sr)

        # Compute WER of this segment vs reference
        try:
            seg_wer = wer(ref_clean, seg_text.lower().strip())
            seg_wer = float(np.clip(seg_wer, 0.0, 1.0))
        except Exception:
            seg_wer = 0.5

        # Label: error if WER > 20%
        label = 1 if seg_wer > 0.20 else 0

        records.append((feats, label, seg_wer, seg_text, word_confs))

    return records

print("✅ Sample processor defined")

In [ ]:
# ── Build training dataset from LibriSpeech + TED-LIUM ────────────────────

all_records = []

print(f"Processing {N_LIBRISPEECH} LibriSpeech samples...")
for i, sample in enumerate(librispeech_ds):
    if i >= N_LIBRISPEECH:
        break
    try:
        audio = sample["audio"]
        arr   = np.array(audio["array"], dtype=np.float32)
        sr    = audio["sampling_rate"]
        ref   = sample.get("text", sample.get("transcription", ""))
        if not ref:
            continue
        recs = process_sample(arr, sr, ref)
        all_records.extend(recs)
        if (i + 1) % 10 == 0:
            print(f"  LibriSpeech: {i+1}/{N_LIBRISPEECH} done, {len(all_records)} segments so far")
    except Exception as e:
        continue

print(f"\nProcessing {N_TEDLIUM} TED-LIUM samples...")
for i, sample in enumerate(tedlium_ds):
    if i >= N_TEDLIUM:
        break
    try:
        audio = sample["audio"]
        arr   = np.array(audio["array"], dtype=np.float32)
        sr    = audio["sampling_rate"]
        ref   = sample.get("text", sample.get("transcription", ""))
        if not ref:
            continue
        recs = process_sample(arr, sr, ref)
        all_records.extend(recs)
        if (i + 1) % 10 == 0:
            print(f"  TED-LIUM: {i+1}/{N_TEDLIUM} done, {len(all_records)} segments so far")
    except Exception as e:
        continue

print(f"\n✅ Total segments collected: {len(all_records)}")
error_count   = sum(1 for r in all_records if r[1] == 1)
correct_count = len(all_records) - error_count
print(f"   Correct segments (WER ≤ 20%): {correct_count}")
print(f"   Error segments   (WER > 20%): {error_count}")

---
## Section 6 — Train LightGBM Trust Score Classifier

In [ ]:
# ── Prepare feature matrix ─────────────────────────────────────────────────

FEATURE_COLS = [
    "conf_mean", "conf_min", "conf_std", "low_conf_ratio",
    "snr_db", "log_perplexity", "avg_word_len", "unique_ratio",
    "repetition_ratio", "jargon_count", "jargon_ratio",
    "segment_length", "n_words"
]

rows = []
for feats, label, seg_wer, seg_text, word_confs in all_records:
    row = {col: feats.get(col, 0.0) for col in FEATURE_COLS}
    row["label"]   = label
    row["seg_wer"]  = seg_wer
    rows.append(row)

df = pd.DataFrame(rows)
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['label'].value_counts()}")
print(f"\nMean WER of error segments: {df[df.label==1]['seg_wer'].mean():.3f}")
print(f"Mean WER of clean segments: {df[df.label==0]['seg_wer'].mean():.3f}")
df.describe()

In [ ]:
# ── Train / test split ─────────────────────────────────────────────────────

X = df[FEATURE_COLS].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

# ── LightGBM classifier ────────────────────────────────────────────────────
lgb_params = {
    "objective":        "binary",
    "metric":           "auc",
    "n_estimators":     300,
    "learning_rate":    0.05,
    "num_leaves":       31,
    "max_depth":        6,
    "min_child_samples": 10,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "reg_alpha":        0.1,
    "reg_lambda":       0.1,
    "class_weight":     "balanced",
    "random_state":     42,
    "n_jobs":           -1,
    "verbose":          -1
}

clf = lgb.LGBMClassifier(**lgb_params)
clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(50)]
)

print("✅ LightGBM trained")

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────

y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("=" * 55)
print("         TRUST SCORE CLASSIFIER — EVALUATION")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Correct", "Error"]))
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}")

# Feature importance
print("\n── Feature Importance ──")
feat_imp = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
for feat, imp in feat_imp.items():
    bar = "█" * int(imp / feat_imp.max() * 25)
    print(f"  {feat:<22} {bar} {imp}")

---
## Section 7 — Core Scoring Engine

This is the function that runs on any video/audio and produces the full output.

In [ ]:
# ── Trust bucket thresholds ────────────────────────────────────────────────
# Trust score = 1 - P(error)  →  higher is better
TRUST_HIGH   = 0.75   # ≥ 75% trust → GREEN
TRUST_MEDIUM = 0.50   # 50–75%     → YELLOW
                       # < 50%      → RED

def score_trust(features_dict: dict) -> dict:
    """Run LightGBM on one segment's features. Return trust score + bucket."""
    feat_vec = np.array([[features_dict.get(col, 0.0) for col in FEATURE_COLS]])
    error_prob = clf.predict_proba(feat_vec)[0][1]
    trust      = 1.0 - error_prob

    if trust >= TRUST_HIGH:
        bucket, color = "HIGH",   "#22c55e"  # solid green
    elif trust >= TRUST_MEDIUM:
        bucket, color = "MEDIUM", "#f59e0b"  # solid amber
    else:
        bucket, color = "LOW",    "#ef4444"  # solid red

    return {"trust": trust, "error_prob": error_prob, "bucket": bucket, "color": color}


def flag_reasons(features_dict: dict, trust_result: dict) -> list:
    """Return concrete, specific reasons why a segment was flagged."""
    reasons = []
    if features_dict["conf_min"] < 0.35:
        reasons.append(f"Low acoustic confidence (min word confidence: {features_dict['conf_min']:.0%})")
    if features_dict["log_perplexity"] > math.log(150):
        reasons.append(f"Unusual phrasing detected (perplexity: {math.exp(features_dict['log_perplexity']):.0f})")
    if features_dict["jargon_ratio"] > 0.15:
        reasons.append(f"Domain jargon heavy ({features_dict['jargon_count']:.0f} specialist terms)")
    if features_dict["repetition_ratio"] > 0.1:
        reasons.append("Repeated words — possible ASR stutter")
    if features_dict["snr_db"] < 10:
        reasons.append(f"Noisy audio (SNR: {features_dict['snr_db']:.1f} dB)")
    if features_dict["low_conf_ratio"] > 0.4:
        reasons.append(f"{features_dict['low_conf_ratio']:.0%} of words below confidence threshold")
    return reasons


def identify_wrong_words(text: str, word_confidences: list, threshold: float = 0.45) -> list:
    """
    Return list of (word, is_wrong) pairs using Whisper word-level probability.
    is_wrong = True if confidence < threshold.
    """
    words = text.strip().split()
    if not word_confidences or len(word_confidences) != len(words):
        # Pad or truncate confidences
        confs = (word_confidences + [0.5] * len(words))[:len(words)]
    else:
        confs = word_confidences
    return [(w, c < threshold, c) for w, c in zip(words, confs)]


print("✅ Scoring engine ready")

In [ ]:
# ── Main transcription + scoring function ─────────────────────────────────

def transcribe_and_score(audio_path: str) -> dict:
    """
    Given path to an audio/video file:
    1. Loads audio (via librosa, works for mp3/wav/m4a/mp4)
    2. Runs Whisper with word timestamps
    3. Scores every segment
    4. Returns structured results
    """
    # Load audio
    arr, sr = librosa.load(audio_path, sr=16000, mono=True)
    arr = arr.astype(np.float32)
    if np.max(np.abs(arr)) > 0:
        arr /= np.max(np.abs(arr))

    # Whisper transcription
    result = asr_model.transcribe(
        arr,
        word_timestamps=True,
        verbose=False,
        temperature=0.0
    )

    segments_out = []
    total_words        = 0
    total_wrong_words  = 0

    for seg in result.get("segments", []):
        text = seg.get("text", "").strip()
        if not text:
            continue

        words_data = seg.get("words", [])
        word_confs = []
        for w in words_data:
            lp = w.get("probability", None)
            if lp is None:
                lp = math.exp(w.get("logprob", -1.0))
            word_confs.append(float(np.clip(lp, 0.0, 1.0)))

        start_s = seg.get("start", 0.0)
        end_s   = seg.get("end", 0.0)
        start_i = int(start_s * sr)
        end_i   = int(end_s   * sr)
        seg_audio = arr[start_i:end_i] if end_i > start_i else arr

        feats  = extract_features(text, word_confs, seg_audio, sr)
        trust  = score_trust(feats)
        reasons = flag_reasons(feats, trust)
        word_annotations = identify_wrong_words(text, word_confs)

        n_words_seg  = len(text.split())
        n_wrong_seg  = sum(1 for _, is_wrong, _ in word_annotations if is_wrong)
        total_words       += n_words_seg
        total_wrong_words += n_wrong_seg

        segments_out.append({
            "start":             start_s,
            "end":               end_s,
            "text":              text,
            "word_annotations":  word_annotations,
            "n_words":           n_words_seg,
            "n_wrong":           n_wrong_seg,
            "trust_score":       trust["trust"],
            "error_prob":        trust["error_prob"],
            "bucket":            trust["bucket"],
            "color":             trust["color"],
            "reasons":           reasons,
            "features":          feats,
        })

    # Overall summary
    n_segs    = len(segments_out)
    n_flagged = sum(1 for s in segments_out if s["bucket"] == "LOW")
    n_medium  = sum(1 for s in segments_out if s["bucket"] == "MEDIUM")
    overall_accuracy = 1.0 - (total_wrong_words / total_words) if total_words > 0 else 1.0

    # Compliance risk
    flag_ratio = n_flagged / n_segs if n_segs > 0 else 0
    if flag_ratio >= 0.35:
        compliance_risk = "HIGH"
    elif flag_ratio >= 0.15:
        compliance_risk = "MEDIUM"
    else:
        compliance_risk = "LOW"

    return {
        "segments":          segments_out,
        "total_words":       total_words,
        "total_wrong":       total_wrong_words,
        "overall_accuracy":  overall_accuracy,
        "n_segments":        n_segs,
        "n_flagged":         n_flagged,
        "n_medium":          n_medium,
        "compliance_risk":   compliance_risk,
    }

print("✅ Main scoring pipeline ready")

---
## Section 8 — Video / YouTube Ingestion

In [ ]:
def download_youtube_audio(url: str, output_dir: str = "/tmp") -> str:
    """
    Download audio from a YouTube URL (or any yt-dlp-supported platform).
    Returns path to the downloaded .wav file.
    """
    out_template = os.path.join(output_dir, "yt_audio.%(ext)s")
    wav_path     = os.path.join(output_dir, "yt_audio.wav")

    # Remove previous if exists
    for f in Path(output_dir).glob("yt_audio.*"):
        f.unlink()

    cmd = [
        "yt-dlp",
        "-x",                           # Extract audio only
        "--audio-format", "wav",         # Output as wav
        "--audio-quality", "0",          # Best quality
        "--ffmpeg-location", "/usr/bin",
        "-o", out_template,
        url
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"yt-dlp failed:\n{result.stderr}")

    if not os.path.exists(wav_path):
        # yt-dlp may have saved with slightly different name
        candidates = list(Path(output_dir).glob("yt_audio.*"))
        if not candidates:
            raise FileNotFoundError("yt-dlp ran but no audio file found")
        wav_path = str(candidates[0])

    return wav_path


def extract_audio_from_video(video_path: str, output_dir: str = "/tmp") -> str:
    """
    Extract audio from any uploaded video file (mp4, mkv, avi, webm, mov).
    Returns path to .wav file.
    """
    wav_path = os.path.join(output_dir, "uploaded_audio.wav")
    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-ar", "16000",   # 16kHz
        "-ac", "1",       # Mono
        "-vn",            # No video
        wav_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{result.stderr}")
    return wav_path


print("✅ Video/YouTube ingestion functions ready")

---
## Section 9 — Output Renderer

Builds the HTML output with color-coded segments, word-level error highlighting, and the session summary.

In [ ]:
def render_output_html(results: dict) -> str:
    """
    Render full SignalPath output as HTML.
    - Session summary block
    - Per-segment cards with word-level highlighting
    """
    segs             = results["segments"]
    total_words      = results["total_words"]
    total_wrong      = results["total_wrong"]
    accuracy         = results["overall_accuracy"]
    n_segs           = results["n_segments"]
    n_flagged        = results["n_flagged"]
    n_medium         = results["n_medium"]
    compliance_risk  = results["compliance_risk"]

    risk_color = {"LOW": "#22c55e", "MEDIUM": "#f59e0b", "HIGH": "#ef4444"}[compliance_risk]
    acc_pct    = accuracy * 100

    html = f"""
<style>
  body {{ font-family: 'IBM Plex Mono', 'Courier New', monospace; background: #0f0f0f; color: #e5e5e5; margin: 0; padding: 0; }}
  .sp-wrap {{ max-width: 900px; margin: 0 auto; padding: 24px 16px; }}

  /* Session Summary */
  .summary-block {{
    background: #1a1a1a;
    border: 1px solid #2a2a2a;
    border-radius: 4px;
    padding: 20px 24px;
    margin-bottom: 28px;
  }}
  .summary-title {{
    font-size: 11px;
    letter-spacing: 0.15em;
    color: #666;
    text-transform: uppercase;
    margin-bottom: 16px;
  }}
  .summary-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));
    gap: 16px;
  }}
  .metric {{ }}
  .metric-val {{
    font-size: 26px;
    font-weight: 700;
    letter-spacing: -0.02em;
    line-height: 1;
  }}
  .metric-label {{
    font-size: 11px;
    color: #888;
    margin-top: 4px;
    text-transform: uppercase;
    letter-spacing: 0.08em;
  }}
  .risk-badge {{
    display: inline-block;
    background: {risk_color}20;
    color: {risk_color};
    border: 1px solid {risk_color}60;
    border-radius: 3px;
    padding: 2px 10px;
    font-size: 13px;
    font-weight: 700;
    letter-spacing: 0.08em;
  }}

  /* Legend */
  .legend {{
    display: flex;
    gap: 20px;
    margin-bottom: 20px;
    font-size: 12px;
    color: #888;
  }}
  .legend-dot {{
    display: inline-block;
    width: 10px; height: 10px;
    border-radius: 50%;
    margin-right: 6px;
    vertical-align: middle;
  }}

  /* Segments */
  .segment {{
    background: #141414;
    border-left: 3px solid var(--seg-color);
    padding: 14px 16px;
    margin-bottom: 10px;
    border-radius: 0 3px 3px 0;
  }}
  .seg-header {{
    display: flex;
    justify-content: space-between;
    align-items: baseline;
    margin-bottom: 10px;
    flex-wrap: wrap;
    gap: 8px;
  }}
  .seg-time {{ font-size: 11px; color: #555; letter-spacing: 0.04em; }}
  .seg-stats {{ font-size: 12px; color: #aaa; }}
  .seg-trust {{
    font-size: 12px;
    font-weight: 700;
    color: var(--seg-color);
    letter-spacing: 0.04em;
  }}

  /* Caption text */
  .caption-text {{
    font-size: 15px;
    line-height: 1.7;
    letter-spacing: 0.01em;
    margin-bottom: 10px;
  }}
  .word-ok  {{ color: #e5e5e5; }}
  .word-err {{
    color: #ef4444;
    text-decoration: underline;
    text-decoration-style: wavy;
    text-underline-offset: 3px;
  }}

  /* Wrong word count */
  .word-count {{
    font-size: 12px;
    color: #666;
    margin-bottom: 8px;
  }}
  .word-count strong {{ color: #ef4444; }}

  /* Reasons */
  .reasons {{ display: flex; flex-wrap: wrap; gap: 6px; }}
  .reason-tag {{
    background: #1f1f1f;
    border: 1px solid #2f2f2f;
    border-radius: 3px;
    padding: 2px 8px;
    font-size: 11px;
    color: #f59e0b;
    letter-spacing: 0.02em;
  }}
</style>

<div class="sp-wrap">

<!-- ── SESSION SUMMARY ────────────────────────────────── -->
<div class="summary-block">
  <div class="summary-title">SignalPath — Session Report</div>
  <div class="summary-grid">
    <div class="metric">
      <div class="metric-val" style="color: {'#22c55e' if acc_pct >= 80 else '#f59e0b' if acc_pct >= 60 else '#ef4444'}">{acc_pct:.1f}%</div>
      <div class="metric-label">Caption accuracy</div>
    </div>
    <div class="metric">
      <div class="metric-val">{total_wrong} <span style="font-size:14px;color:#666">/ {total_words}</span></div>
      <div class="metric-label">Words likely wrong</div>
    </div>
    <div class="metric">
      <div class="metric-val" style="color:#ef4444">{n_flagged}</div>
      <div class="metric-label">Segments flagged</div>
    </div>
    <div class="metric">
      <div class="metric-val" style="color:#f59e0b">{n_medium}</div>
      <div class="metric-label">Segments at risk</div>
    </div>
    <div class="metric">
      <div class="metric-val">{n_segs}</div>
      <div class="metric-label">Total segments</div>
    </div>
    <div class="metric">
      <div class="metric-label" style="margin-top:6px">Compliance risk</div>
      <div class="risk-badge">{compliance_risk}</div>
    </div>
  </div>
</div>

<!-- ── LEGEND ─────────────────────────────────────────── -->
<div class="legend">
  <span><span class="legend-dot" style="background:#22c55e"></span>High trust (≥75%)</span>
  <span><span class="legend-dot" style="background:#f59e0b"></span>Medium trust (50–75%)</span>
  <span><span class="legend-dot" style="background:#ef4444"></span>Low trust (&lt;50%) — flagged</span>
  <span style="color:#ef4444">underline</span><span> = likely wrong word</span>
</div>

<!-- ── SEGMENTS ───────────────────────────────────────── -->
"""

    for seg in segs:
        color        = seg["color"]
        trust_pct    = seg["trust_score"] * 100
        n_w          = seg["n_words"]
        n_err        = seg["n_wrong"]
        n_correct    = n_w - n_err
        start_fmt    = f"{int(seg['start']//60):02d}:{seg['start']%60:05.2f}"
        end_fmt      = f"{int(seg['end']//60):02d}:{seg['end']%60:05.2f}"

        # Build word-annotated caption
        words_html = []
        for word, is_wrong, conf in seg["word_annotations"]:
            cls   = "word-err" if is_wrong else "word-ok"
            title = f"conf={conf:.0%}"
            words_html.append(f'<span class="{cls}" title="{title}">{word}</span>')
        caption_html = " ".join(words_html)

        # Reasons
        reasons_html = ""
        if seg["reasons"]:
            tags = " ".join(f'<span class="reason-tag">⚠ {r}</span>' for r in seg["reasons"])
            reasons_html = f'<div class="reasons">{tags}</div>'

        html += f"""
<div class="segment" style="--seg-color:{color}">
  <div class="seg-header">
    <span class="seg-time">{start_fmt} → {end_fmt}</span>
    <span class="seg-stats">{n_err} wrong / {n_w} words</span>
    <span class="seg-trust">TRUST {trust_pct:.0f}% [{seg['bucket']}]</span>
  </div>
  <div class="caption-text">{caption_html}</div>
  <div class="word-count">
    <strong>{n_err} word{'s' if n_err != 1 else ''} likely wrong</strong> out of {n_w} total
    &nbsp;·&nbsp; {n_correct} correct
  </div>
  {reasons_html}
</div>
"""

    html += "</div>"  # close sp-wrap
    return html


print("✅ Output renderer ready")

---
## Section 10 — Gradio Frontend

In [ ]:
def run_pipeline(youtube_url: str, uploaded_file) -> tuple:
    """
    Gradio handler.
    Accepts either a YouTube URL or an uploaded file.
    Returns (html_output, status_message)
    """
    status_log = []

    try:
        # Determine source
        if youtube_url and youtube_url.strip():
            url = youtube_url.strip()
            status_log.append(f"Source: YouTube URL → {url}")
            status_log.append("Downloading audio...")
            audio_path = download_youtube_audio(url)
            status_log.append(f"Audio saved: {audio_path}")

        elif uploaded_file is not None:
            file_path = uploaded_file.name if hasattr(uploaded_file, 'name') else str(uploaded_file)
            status_log.append(f"Source: Uploaded file → {os.path.basename(file_path)}")

            ext = Path(file_path).suffix.lower()
            if ext in [".mp4", ".mkv", ".avi", ".webm", ".mov", ".flv"]:
                status_log.append("Extracting audio from video...")
                audio_path = extract_audio_from_video(file_path)
            else:
                # Already audio (wav, mp3, m4a, flac, ogg)
                audio_path = file_path
            status_log.append(f"Audio ready: {audio_path}")

        else:
            return "<p style='color:#ef4444;font-family:monospace'>Please provide a YouTube URL or upload a file.</p>", "No input provided."

        # Run scoring
        status_log.append("Running Whisper ASR...")
        results = transcribe_and_score(audio_path)
        status_log.append(f"Transcribed {results['n_segments']} segments, {results['total_words']} words")

        # Render
        html_out = render_output_html(results)

        summary = (
            f"✅ Done\n"
            f"Accuracy: {results['overall_accuracy']*100:.1f}% | "
            f"Wrong words: {results['total_wrong']}/{results['total_words']} | "
            f"Flagged segments: {results['n_flagged']}/{results['n_segments']} | "
            f"Compliance risk: {results['compliance_risk']}\n"
            + "\n".join(status_log)
        )
        return html_out, summary

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        err_html = f"""<div style='font-family:monospace;color:#ef4444;padding:20px'>
            <strong>Error:</strong><br>{str(e)}<br><pre style='font-size:11px;color:#888'>{tb}</pre></div>"""
        return err_html, f"❌ Error: {str(e)}\n" + "\n".join(status_log)


print("✅ Gradio pipeline handler ready")

In [ ]:
# ── Launch Gradio Dashboard ────────────────────────────────────────────────

DARK_CSS = """
body, .gradio-container { background: #0f0f0f !important; color: #e5e5e5 !important; }
.dark { background: #0f0f0f !important; }
footer { display: none !important; }
"""

with gr.Blocks(css=DARK_CSS, theme=gr.themes.Base()) as demo:

    gr.HTML("""
    <div style="font-family:'IBM Plex Mono',monospace;padding:24px 0 8px">
      <div style="font-size:11px;letter-spacing:0.18em;color:#555;text-transform:uppercase;margin-bottom:6px">
        Real-Time Caption Intelligence
      </div>
      <div style="font-size:28px;font-weight:700;letter-spacing:-0.02em;color:#e5e5e5">
        SignalPath
      </div>
      <div style="font-size:13px;color:#888;margin-top:6px">
        Caption trust scoring · Word-level error detection · Accessibility gap analysis
      </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")

            youtube_input = gr.Textbox(
                label="YouTube URL (or any yt-dlp supported URL)",
                placeholder="https://www.youtube.com/watch?v=...",
                lines=1
            )
            gr.Markdown("**— or —**", elem_id="or_divider")
            file_input = gr.File(
                label="Upload video or audio file",
                file_types=[".mp4", ".mkv", ".avi", ".webm", ".mov",
                            ".wav", ".mp3", ".m4a", ".flac", ".ogg"]
            )
            run_btn = gr.Button("▶  Run SignalPath", variant="primary")

            gr.Markdown("""
            **How it works:**
            1. Whisper transcribes the audio into segments
            2. Each word gets a confidence score from Whisper
            3. LightGBM classifier predicts segment trust (acoustic + linguistic + domain features)
            4. DistilGPT2 checks linguistic plausibility
            5. Results displayed with word-level error marking
            """)

            status_box = gr.Textbox(
                label="Run log",
                lines=6,
                interactive=False
            )

        with gr.Column(scale=2):
            gr.Markdown("### Caption Output")
            output_html = gr.HTML(
                value="<div style='font-family:monospace;color:#444;padding:40px;text-align:center'>Output will appear here after processing.</div>"
            )

    run_btn.click(
        fn=run_pipeline,
        inputs=[youtube_input, file_input],
        outputs=[output_html, status_box]
    )

# share=True gives you a public URL that works outside Colab
demo.launch(share=True, debug=False)
print("✅ Gradio dashboard launched — check the public URL above")

---
## Section 11 — Quick Test (No UI)

Run this cell to test the pipeline directly on a YouTube URL without opening the Gradio UI.

In [ ]:
# ── Direct test — paste any YouTube URL here ──────────────────────────────

TEST_URL = "https://www.youtube.com/watch?v=arj7oStGLkU"  # TED Talk: Do schools kill creativity

print(f"Testing on: {TEST_URL}")
print("Downloading audio...")
audio_path = download_youtube_audio(TEST_URL)
print(f"Audio: {audio_path}")

print("\nRunning transcription + scoring...")
results = transcribe_and_score(audio_path)

print("\n" + "=" * 55)
print("  SIGNALPATH SESSION REPORT")
print("=" * 55)
print(f"  Caption accuracy:   {results['overall_accuracy']*100:.1f}%")
print(f"  Words likely wrong: {results['total_wrong']} / {results['total_words']}")
print(f"  Segments total:     {results['n_segments']}")
print(f"  Flagged (LOW):      {results['n_flagged']}")
print(f"  At risk (MEDIUM):   {results['n_medium']}")
print(f"  Compliance risk:    {results['compliance_risk']}")
print("=" * 55)

print("\n── Per-Segment Breakdown ──")
for i, seg in enumerate(results["segments"][:10]):  # show first 10
    print(f"\n[{i+1}] {int(seg['start']//60):02d}:{seg['start']%60:05.2f} → {int(seg['end']//60):02d}:{seg['end']%60:05.2f}")
    print(f"     Trust: {seg['trust_score']*100:.0f}% [{seg['bucket']}]  |  {seg['n_wrong']} wrong / {seg['n_words']} words")
    print(f"     Text: {seg['text'][:100]}")
    if seg["reasons"]:
        for r in seg["reasons"]:
            print(f"     ⚠ {r}")